# Proyecto de Aprendizaje de Máquina — Parte 2
### Deep Learning y aumentación de datos para clasificación multilingüe de sentimientos




## 1. Objetivo y diseño experimental

Se clasifican reseñas en **negativo**, **neutral** o **positivo**. Hay textos en español, inglés, francés, portugués y con cambio de idioma, además de ironía, negaciones y concesiones. Se usa **accuracy** como métrica principal y un CSV final con dos columnas: id,answer.

Se crea un conjunto de validación fijo (15 %) a partir de train.csv. Se entrena un SVM con TF-IDF exclusivamente como referencia diagnóstica, seguido de tres redes profundas: mDeBERTa sin aumentación, el mismo mDeBERTa con aumentación y XLM-RoBERTa preentrenado en sentimientos con aumentación. Los tres utilizan la misma partición, semillas y protocolo de evaluación. Se analiza el desempeño por idioma, se comparan ensambles de probabilidades y se exporta el modelo ganador. Opcionalmente se reentrena con todo el conjunto etiquetado.

Aumentación: edición ortográfica de tildes, mayúsculas y separadores con probabilidad del 40 %, sin introducir nuevas palabras ni tocar directamente la polaridad. Se mide su efecto mediante una comparación controlada con y sin aumentación.

Referencia: las condiciones completas, el análisis por idioma y los entregables se encuentran en el enunciado del curso. El notebook mantiene en Markdown la explicación de cada celda de código.


In [2]:
import os, sys, platform, subprocess
import torch
print('Python:', sys.version.split()[0], '| Sistema:', platform.system())
print('PyTorch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Activa una GPU en Entorno de ejecución > Cambiar tipo de entorno de ejecución > T4 GPU, conéctala y vuelve a ejecutar esta celda.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM total (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
subprocess.run(['nvidia-smi'], check=False)


Python: 3.13.16 | Sistema: Linux
PyTorch: 2.14.1+cu130
CUDA disponible: True
GPU: Tesla T4
VRAM total (GB): 14.56


CompletedProcess(args=['nvidia-smi'], returncode=0)

### 2.1. Instalación y configuración

Se instalan transformers, accelerate, sentencepiece, langdetect, protobuf y utilidades manteniendo PyTorch/CUDA de Colab.

El entrenamiento utiliza precisión mixta FP16, acumulación de gradientes, gradient checkpointing, límite de longitud y parada temprana para administrar la memoria disponible. Se fijan semillas para reducir la variabilidad.



In [ ]:
import sys, subprocess, importlib.metadata, importlib.util

def instalar_colab():
    dependencias = [
        ('transformers', '4.57.3', 'transformers==4.57.3'),
        ('accelerate', None, 'accelerate>=1.2,<2'),
        ('sentencepiece', None, 'sentencepiece>=0.2'),
        ('langdetect', '1.0.9', 'langdetect==1.0.9'),
        ('protobuf', None, 'protobuf>=4.25,<7'),
        ('safetensors', None, 'safetensors>=0.4'),
    ]
    faltantes = []
    for paquete, version, requerimiento in dependencias:
        try:
            instalada = importlib.metadata.version(paquete)
        except importlib.metadata.PackageNotFoundError:
            instalada = None
        if instalada is None or (version and instalada != version):
            faltantes.append(requerimiento)
    if faltantes:
        print('Instalando:', ', '.join(faltantes))
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *faltantes])
    print('Dependencias listas. No se reinstaló torch.')

instalar_colab()

import os, gc, json, math, random, re, unicodedata, shutil, contextlib, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from tqdm.auto import tqdm
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, log_loss
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
from transformers import get_cosine_schedule_with_warmup

PROJECT_DIR = Path('/content')
DATA_DIR = PROJECT_DIR
OUTPUT_DIR = PROJECT_DIR / 'salidas'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REUTILIZAR_EXPERIMENTOS = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DetectorFactory.seed = SEED
torch.backends.cudnn.benchmark = False
DEVICE = torch.device('cuda')
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
USAR_FP16 = True
AMP_DTYPE = torch.float16
MAX_LENGTH = 192
BATCH_SIZE = 4 if VRAM_GB >= 14 else 2
EVAL_BATCH_SIZE = 8 if VRAM_GB >= 14 else 4
GRAD_ACCUM = 4 if VRAM_GB >= 14 else 8
USE_GRADIENT_CHECKPOINTING = True
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
MAX_EPOCHS = 3
PATIENCE = 2
AUG_PROB = 0.40
VAL_FRAC = 0.15
REENTRENAR_FINAL = False
EMPAQUETAR_MODELO = True
DESCARGAR_SUBMISSION = True
DESCARGAR_MODELO_ZIP = False

LABELS = ['negativo','neutral','positivo']
LABEL2ID = {clase:i for i,clase in enumerate(LABELS)}
ID2LABEL = {i:clase for clase,i in LABEL2ID.items()}

def contexto_amp():
    return torch.autocast(device_type='cuda', dtype=AMP_DTYPE) if USAR_FP16 else contextlib.nullcontext()

print('GPU:', torch.cuda.get_device_name(0), '| memoria:', round(VRAM_GB,1),'GB')
print('MAX_LENGTH:',MAX_LENGTH,'| batch:',BATCH_SIZE,'| acumulación:',GRAD_ACCUM, '| FP16:',USAR_FP16)
print('Archivos CSV esperados en:', DATA_DIR)
print('Resultados temporales en:', OUTPUT_DIR)
print('IMPORTANTE: descarga los resultados antes de que Colab desconecte la sesión.')


Instalando: transformers==4.57.3, langdetect==1.0.9


### 2.2. Prueba de entrenamiento GPU

Se crea un pequeño BERT con pesos inicializados aleatoriamente y se completa una pasada de entrenamiento en CUDA usando precisión mixta. Esta prueba comprueba que el entorno es apto para operaciones de redes profundas (forward, backward y actualización de parámetros); no mide accuracy ni sustituye los experimentos posteriores.


In [ ]:
from transformers import BertConfig, BertForSequenceClassification
cfg_prueba = BertConfig(vocab_size=256, hidden_size=64, num_hidden_layers=1,
                       num_attention_heads=4, intermediate_size=128,
                       max_position_embeddings=64, num_labels=3, attn_implementation='eager')
mini_modelo = BertForSequenceClassification(cfg_prueba).to(DEVICE)
mini_modelo.train()
mini_opt = torch.optim.AdamW(mini_modelo.parameters(), lr=2e-4)
mini_ids = torch.randint(0, 256, (2, 24), dtype=torch.long, device=DEVICE)
mini_mask = torch.ones_like(mini_ids)
mini_y = torch.tensor([0, 2], dtype=torch.long, device=DEVICE)
mini_scaler = torch.amp.GradScaler('cuda', enabled=USAR_FP16)
mini_opt.zero_grad(set_to_none=True)
with contexto_amp():
    mini_logits = mini_modelo(input_ids=mini_ids, attention_mask=mini_mask).logits
    mini_loss = F.cross_entropy(mini_logits.float(), mini_y)
mini_scaler.scale(mini_loss).backward()
mini_scaler.step(mini_opt)
mini_scaler.update()
print('Entrenamiento GPU de prueba: OK | loss:', round(mini_loss.item(), 5))
del mini_modelo, mini_opt, mini_ids, mini_mask, mini_y, mini_scaler
import gc
gc.collect()
torch.cuda.empty_cache()


## 3. Carga de los CSV subidos directamente al notebook

Los tres archivos deben estar en el panel **Archivos** de Colab, en /content/train.csv, /content/eval.csv y /content/sample_submission.csv.

Se comprueba el formato esperado. Se mantiene el orden original de los identificadores para que la entrega a Kaggle cumpla `id,answer`. Los textos de `eval.csv` no se utilizan para entrenar.


In [ ]:
DATA_DIR = Path('/content')
NECESARIOS = ('train.csv','eval.csv','sample_submission.csv')
faltantes = [f for f in NECESARIOS if not (DATA_DIR / f).is_file()]
if faltantes:
    raise FileNotFoundError(
        f'Faltan {faltantes} en {DATA_DIR}. Abre el panel izquierdo de Colab (Archivos), '
        'sube train.csv, eval.csv y sample_submission.csv directamente a /content y vuelve a ejecutar esta celda. ')

train_df = pd.read_csv(DATA_DIR/'train.csv')
eval_df = pd.read_csv(DATA_DIR/'eval.csv')
plantilla = pd.read_csv(DATA_DIR/'sample_submission.csv')
assert list(train_df.columns) == ['id','text','label']
assert list(eval_df.columns) == ['id','text']
assert list(plantilla.columns) == ['id','answer']
for nombre, df in [('train',train_df),('eval',eval_df)]:
    assert not df.isna().any().any(), f'{nombre}: datos faltantes'
    assert df.id.is_unique, f'{nombre}: IDs repetidos'
    assert df.text.map(lambda x: isinstance(x,str) and bool(x.strip())).all()
assert set(train_df.label.unique()) == set(LABELS)
assert plantilla.id.equals(eval_df.id), 'Los IDs de eval y plantilla deben coincidir en orden'
assert len(train_df) == 20466 and len(eval_df) == 6781, 'Revisa que los archivos correspondan a la Parte 2.'
print('Datos cargados desde:',DATA_DIR)
print('Entrenamiento:',train_df.shape,'| Evaluación:',eval_df.shape)
display(train_df.head(3))


## 4. Exploración de la longitud y del desbalance de clases

Se examina el tamaño de las reseñas y la frecuencia de las tres clases. Estas medidas sirven para elegir `MAX_LENGTH` y detectar un posible desbalance sin consultar las etiquetas ocultas de `eval.csv`. No se eliminan stopwords ni signos de negación, ya que hacerlo destruiría información importante para el sentimiento.

In [ ]:
train_df['n_palabras'] = train_df['text'].str.split().str.len()
train_df['n_caracteres'] = train_df['text'].str.len()
print('Longitud (palabras):')
display(train_df['n_palabras'].describe(percentiles=[.5, .9, .95, .99]).to_frame())
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
train_df['label'].value_counts().reindex(LABELS).plot(kind='bar', ax=axes[0], rot=0)
axes[0].set_title('Distribución de clases')
axes[0].set_ylabel('Reseñas')
train_df['n_palabras'].hist(bins=35, ax=axes[1])
axes[1].set_title('Longitud de las reseñas')
axes[1].set_xlabel('Palabras')
plt.tight_layout()
plt.show()

## 5. Etiquetado auxiliar del idioma (solo para estratificar y analizar errores)

Se usa `langdetect` con semilla fija para etiquetar **aproximadamente** cada texto como español, inglés, francés o portugués. Una regla adicional, basada en palabras distintivas de ambas lenguas, marca posibles reseñas mixtas español–inglés. Este grupo es heurístico y puede contener falsos positivos o negativos: no se presenta como anotación humana perfecta. El idioma **no** entra como una etiqueta de sentimiento ni se usa para entrenar sobre `eval`.

El análisis de idiomas busca evitar que una validación dominada por español oculte fallas en las otras lenguas.

In [ ]:
ES_PISTAS = {'compré','compre','llegó','llego','aunque','entonces','tampoco','pero','porque','siempre',
             'envío','envio','pantalla','batería','bateria','devolví','devolvi','pedido','todavía','todavia'}
EN_PISTAS = {'bought','ordered','arrived','because','however','although','battery','screen','wouldn',
             'wouldnt','wasnt','was','were','worked','thought','instead','still','never','return','refund'}

def idioma_aproximado(texto):
    tokens = re.findall(r"[a-záéíóúüñçàâêîôûãõ']+", texto.lower())
    es = sum(token in ES_PISTAS for token in tokens)
    en = sum(token in EN_PISTAS for token in tokens)
    if es >= 2 and en >= 2:
        return 'es_en_mixto'
    try:
        lang = detect(texto[:1200])
    except LangDetectException:
        lang = 'otro'
    return lang if lang in {'es', 'en', 'fr', 'pt'} else 'otro'

train_df['idioma'] = train_df['text'].map(idioma_aproximado)
print('Idiomas aproximados en entrenamiento:')
display(pd.crosstab(train_df.idioma, train_df.label, margins=True))

## 6. Partición fija de entrenamiento y validación

Se estratifica por **clase × idioma aproximado**; si una combinación es demasiado escasa se estratifica solo por clase. Así, todos los experimentos usan exactamente los mismos ejemplos de validación. A partir de este punto, los hiperparámetros, los pesos de las fusiones y la selección del modelo solo se comparan contra esta partición, nunca contra `eval.csv`. La semilla queda explícita.

In [ ]:
estratos = train_df['label'] + '__' + train_df['idioma']
conteos = estratos.value_counts()
estratos = estratos.where(estratos.map(conteos) >= 2, train_df['label'])
indices_train, indices_val = train_test_split(
    np.arange(len(train_df)), test_size=VAL_FRAC, random_state=SEED,
    stratify=estratos,
)
sub_train = train_df.iloc[indices_train].reset_index(drop=True).copy()
sub_val = train_df.iloc[indices_val].reset_index(drop=True).copy()
assert set(sub_train.id).isdisjoint(set(sub_val.id))
assert len(sub_train) + len(sub_val) == len(train_df)
assert set(sub_val.label.unique()) == set(LABELS)

y_val = sub_val.label.map(LABEL2ID).to_numpy(dtype=np.int64)
print('Entrenamiento:', len(sub_train), 'Validación:', len(sub_val))
print('Distribuciones (%):')
display(pd.concat([
    sub_train.label.value_counts(normalize=True).rename('entrenamiento'),
    sub_val.label.value_counts(normalize=True).rename('validacion'),
], axis=1).reindex(LABELS).mul(100).round(2))

## 7. Referencia clásica antes de entrenar las redes profundas

Se calcula una primera referencia **exclusivamente diagnóstica** con TF-IDF de unigramas/bigramas y SVM lineal. Esta referencia no satisface el requisito de deep learning ni podrá ser la solución final. Su utilidad es cuantificar cuánto cuesta capturar negación, concesiones y sentido global con representaciones basadas en n-gramas. Se ajusta con `sub_train` y se evalúa sobre `sub_val`, sin usar `eval.csv`.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

vectorizador = TfidfVectorizer(
    ngram_range=(1, 2), min_df=2, sublinear_tf=True,
    strip_accents='unicode', max_features=160000,
)
X_tr = vectorizador.fit_transform(sub_train['text'])
X_val = vectorizador.transform(sub_val['text'])
svm = LinearSVC(C=1.5, random_state=SEED)
svm.fit(X_tr, sub_train.label)
pred_svm = svm.predict(X_val)
accuracy_svm = accuracy_score(sub_val.label, pred_svm)
print(f'Accuracy de referencia TF-IDF + SVM: {accuracy_svm:.4f}')
print(classification_report(sub_val.label, pred_svm, labels=LABELS))
del X_tr, X_val, vectorizador, svm
gc.collect()

## 8. Aumentación de datos controlada y sensible al significado

La estrategia `aug_superficial` aplica al azar, con probabilidad `AUG_PROB = 0.40` por reseña y por época, **una** transformación pequeña: quitar algunas tildes (imitando escritura informal), modificar la capitalización o sustituir una coma por punto y coma, o viceversa. El texto, su orden, las palabras de sentimiento y las negaciones permanecen. Por construcción no hay traducción automática ni generación de etiquetas sintéticas.

La aumentación es en línea: no se guarda un nuevo CSV, no se incrementa artificialmente el número de iteraciones y se deja intacta la validación. Se compara el mismo modelo con y sin aumentación. Como toda transformación automática, podría alterar algún matiz; por eso se emplean cambios pequeños y se determina su efecto empíricamente.

In [ ]:
PATRON_PALABRA = re.compile(r"\b[\wÀ-ÿ]+\b", flags=re.UNICODE)

def quitar_tildes(palabra):
    return ''.join(c for c in unicodedata.normalize('NFD', palabra)
                   if unicodedata.category(c) != 'Mn')

def aug_superficial(texto, rng):
    salida = texto
    operacion = rng.choice(('tildes', 'capitalizacion', 'puntuacion'))
    if operacion == 'tildes':
        spans = [(m.start(), m.end(), m.group(0)) for m in PATRON_PALABRA.finditer(salida)]
        candidatos = [(a, b, w) for a, b, w in spans if quitar_tildes(w) != w]
        if candidatos:
            a, b, w = rng.choice(candidatos)
            salida = salida[:a] + quitar_tildes(w) + salida[b:]
    elif operacion == 'capitalizacion':
        spans = list(PATRON_PALABRA.finditer(salida))
        if spans:
            m = rng.choice(spans)
            w = m.group(0)
            nuevo = w.lower() if w != w.lower() else w[0].upper() + w[1:]
            salida = salida[:m.start()] + nuevo + salida[m.end():]
    else:
        candidatos = [i for i, c in enumerate(salida) if c in ',;']
        if candidatos:
            i = rng.choice(candidatos)
            salida = salida[:i] + (';' if salida[i] == ',' else ',') + salida[i+1:]
    if salida == texto:
        m = PATRON_PALABRA.search(salida)
        if m:
            palabra = m.group(0)
            nuevo = palabra[0].swapcase() + palabra[1:]
            salida = salida[:m.start()] + nuevo + salida[m.end():]
    return salida

for i in range(3):
    ejemplo = sub_train.iloc[i]
    nuevo = aug_superficial(ejemplo.text, random.Random(SEED+i))
    print('\nEtiqueta original/conservada:', ejemplo.label)
    print('ORIGINAL:', ejemplo.text[:250])
    print('AUMENTADO:', nuevo[:250])

## 9. Dataset, lotes dinámicos e inferencia CUDA

La clase `ResenasDataset` conserva cada texto original y, cuando se solicita aumentación, aplica modificaciones superficiales **en el entrenamiento solamente**. Los textos se tokenizan con padding dinámico al construir cada lote; no se truncan hasta que pasan el máximo `MAX_LENGTH`.

La función `predecir_probabilidades` ejecuta inferencia sobre el checkpoint del modelo y devuelve la probabilidad de cada clase (en el orden negativo, neutral, positivo). En CUDA se utiliza **autocast FP16** para acelerar la inferencia, con conversión de logits a FP32 antes de `softmax` para estabilidad numérica.


In [ ]:
class ResenasDataset(Dataset):
    def __init__(self, df, augmentar=False, semilla=SEED):
        self.textos = df['text'].tolist()
        self.labels = df['label'].map(LABEL2ID).tolist() if 'label' in df else None
        self.augmentar = augmentar
        self.semilla = semilla
        self.epoca = 0

    def set_epoch(self, epoca):
        self.epoca = epoca

    def __len__(self):
        return len(self.textos)

    def __getitem__(self, i):
        texto = self.textos[i]
        if self.augmentar:
            rng = random.Random(self.semilla + self.epoca * 1000003 + i * 11939)
            if rng.random() < AUG_PROB:
                texto = aug_superficial(texto, rng)
        return {'text': texto, 'labels': None if self.labels is None else int(self.labels[i])}


def crear_collator(tokenizer):
    def collator(registros):
        textos = [r['text'] for r in registros]
        entradas = tokenizer(textos, padding=True, truncation=True,
                             max_length=MAX_LENGTH, return_tensors='pt')
        if registros[0]['labels'] is not None:
            entradas['labels'] = torch.tensor([r['labels'] for r in registros], dtype=torch.long)
        return entradas
    return collator


def liberar_gpu(*objetos):
    for obj in objetos:
        del obj
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    elif DEVICE.type == 'mps' and hasattr(torch, 'mps'):
        torch.mps.empty_cache()


def predecir_probabilidades(checkpoint, df):
    tokenizer = AutoTokenizer.from_pretrained(str(checkpoint), use_fast=True)
    modelo = AutoModelForSequenceClassification.from_pretrained(str(checkpoint)).to(DEVICE)
    modelo.eval()
    ds = ResenasDataset(df, augmentar=False)
    loader = DataLoader(ds, batch_size=EVAL_BATCH_SIZE, shuffle=False,
                        num_workers=0, collate_fn=crear_collator(tokenizer))
    bloques = []
    with torch.inference_mode():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items() if k != 'labels'}
            with contexto_amp():
                logits = modelo(**batch).logits
            bloques.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
    resultado = np.concatenate(bloques, axis=0)
    assert resultado.shape == (len(df), len(LABELS))
    assert np.allclose(resultado.sum(axis=1), 1, atol=1e-4)
    del modelo, tokenizer, loader, ds
    liberar_gpu()
    return resultado


## 10. Fine-tuning de transformers en GPU NVIDIA con checkpoints temporales en Colab

Se usan modelos preentrenados públicos, compatibles con las tres clases, y se actualizan sus pesos para las reseñas de la competencia. microsoft/mdeberta-v3-base utiliza un encoder multilingüe; clapAI/roberta-base-multilingual-sentiment usa un XLM-R previamente entrenado para sentimiento. El segundo modelo tiene originalmente las clases negative, neutral, positive, en el mismo orden que nuestro mapeo al español, por lo que su cabeza final puede reutilizarse sin permutar los pesos.

Optimizador AdamW, tasa de aprendizaje 2e-5, scheduler coseno y warmup, tres épocas máximas, acumulación de gradientes, precisión mixta y parada temprana. Cada mejor checkpoint se guarda en /content/salidas al terminar una época; una reejecución con parámetros iguales puede **reutilizar experimentos completados**, pero no reanuda una época interrumpida a la mitad. Se almacena también un resumen de entrenamiento y las versiones de dependencias.


In [ ]:
BASE_MDEBERTA = 'microsoft/mdeberta-v3-base'  # https://huggingface.co/microsoft/mdeberta-v3-base | MIT
BASE_SENTIMIENTO = 'clapAI/roberta-base-multilingual-sentiment'  # Apache-2.0


def guardar_pesos_cpu(modelo, tokenizer, destino):
    """Guardar un checkpoint reproducible sin depender de la GPU temporal."""
    destino = Path(destino)
    destino.mkdir(parents=True, exist_ok=True)
    pesos_cpu = {nombre:t.detach().cpu().contiguous() for nombre,t in modelo.state_dict().items()}
    modelo.save_pretrained(destino, state_dict=pesos_cpu, safe_serialization=True)
    tokenizer.save_pretrained(destino)
    del pesos_cpu


def construir_modelo(base):
    tokenizer = AutoTokenizer.from_pretrained(base, use_fast=True)
    config = AutoConfig.from_pretrained(base)
    originales = {int(k):str(v).lower() for k,v in config.id2label.items()}
    tiene_cabeza = len(originales)==3 and [originales.get(i) for i in range(3)] in [
        ['negative','neutral','positive'], ['negativo','neutral','positivo']]
    modelo = AutoModelForSequenceClassification.from_pretrained(
        base, num_labels=3, ignore_mismatched_sizes=True,
        attn_implementation='eager', id2label=ID2LABEL, label2id=LABEL2ID,
    )
    print(('Reutilizando cabeza de sentimiento: ' if tiene_cabeza else 'Cabeza de clasificación adaptada: ')+base)
    return tokenizer,modelo


def evaluar_epoca(modelo, loader):
    modelo.eval()
    perdida,n=0.0,0
    predicciones,esperados=[],[]
    with torch.inference_mode():
        for lote in loader:
            lote={k:v.to(DEVICE) for k,v in lote.items()}
            y=lote.pop('labels')
            with contexto_amp():
                logits=modelo(**lote).logits
                loss=F.cross_entropy(logits.float(),y)
            perdida+=float(loss.item())*len(y)
            n+=len(y)
            esperados.extend(y.cpu().tolist())
            predicciones.extend(logits.argmax(-1).cpu().tolist())
    return {'accuracy':accuracy_score(esperados,predicciones),'loss':perdida/n}


def ajustar_transformer(base, tag, df_ajuste, df_validacion=None,
                        aumentar=False, epocas=MAX_EPOCHS, parada_temprana=True):
    carpeta = OUTPUT_DIR/('experimentos' if df_validacion is not None else 'modelo_final')/tag
    checkpoint=carpeta/'checkpoint'
    carpeta.mkdir(parents=True,exist_ok=True)
    clave_parametros={
        'base':base,'tag':tag,'aumentacion':bool(aumentar),'epocas':int(epocas),
        'seed':SEED,'max_length':MAX_LENGTH,'batch_size':BATCH_SIZE,'grad_accum':GRAD_ACCUM,
        'lr':LEARNING_RATE,'weight_decay':WEIGHT_DECAY,'aug_prob':AUG_PROB,
        'n_ajuste':len(df_ajuste),'n_validacion':0 if df_validacion is None else len(df_validacion),
        'ids_ajuste_sha256':hashlib.sha256(','.join(map(str,df_ajuste.id.tolist())).encode()).hexdigest(),
        'ids_validacion_sha256':None if df_validacion is None else hashlib.sha256(','.join(map(str,df_validacion.id.tolist())).encode()).hexdigest(),
        'precision_mixta':USAR_FP16,'checkpoint_gradientes':USE_GRADIENT_CHECKPOINTING,
    }
    resumen_path=carpeta/'entrenamiento.json'
    if REUTILIZAR_EXPERIMENTOS and resumen_path.is_file() and (checkpoint/'config.json').is_file():
        try:
            resumen_guardado=json.loads(resumen_path.read_text(encoding='utf-8'))
            pesos_ok=any((checkpoint/n).is_file() for n in ('model.safetensors','pytorch_model.bin'))
            if pesos_ok and resumen_guardado.get('parametros')==clave_parametros:
                print(f'Checkpoint terminado reutilizado: {tag}')
                return resumen_guardado
        except (OSError,ValueError,KeyError):
            pass

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    tokenizer,modelo=construir_modelo(base)
    modelo.to(DEVICE)
    if USE_GRADIENT_CHECKPOINTING and getattr(modelo,'supports_gradient_checkpointing',False):
        try:
            modelo.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
            print('Gradient checkpointing activo')
        except Exception:
            modelo.gradient_checkpointing_enable()
    print('Entrenando:',tag,'| GPU:',torch.cuda.get_device_name(0))
    ds_train=ResenasDataset(df_ajuste,augmentar=aumentar)
    loader_train=DataLoader(ds_train,batch_size=BATCH_SIZE,shuffle=True,
        generator=torch.Generator().manual_seed(SEED),num_workers=0,
        collate_fn=crear_collator(tokenizer))
    loader_val=None
    if df_validacion is not None:
        loader_val=DataLoader(ResenasDataset(df_validacion),batch_size=EVAL_BATCH_SIZE,
            shuffle=False,num_workers=0,collate_fn=crear_collator(tokenizer))

    no_decay=('bias','LayerNorm.weight','layer_norm.weight')
    grupos=[
        {'params':[p for n,p in modelo.named_parameters() if p.requires_grad and not any(x in n for x in no_decay)], 'weight_decay':WEIGHT_DECAY},
        {'params':[p for n,p in modelo.named_parameters() if p.requires_grad and any(x in n for x in no_decay)], 'weight_decay':0.0},
    ]
    optimizador=torch.optim.AdamW(grupos,lr=LEARNING_RATE)
    pasos_epoca=math.ceil(len(loader_train)/GRAD_ACCUM)
    n_pasos=pasos_epoca*epocas
    scheduler=get_cosine_schedule_with_warmup(optimizador,
        num_warmup_steps=max(1,round(n_pasos*0.08)),num_training_steps=n_pasos)
    scaler=torch.amp.GradScaler('cuda',enabled=USAR_FP16)
    historial=[]
    mejor,mejor_loss,mejor_epoca=-np.inf,float('inf'),0
    sin_mejora=0
    for epoca in range(1,epocas+1):
        modelo.train()
        ds_train.set_epoch(epoca)
        optimizador.zero_grad(set_to_none=True)
        suma_loss,total=0.0,0
        torch.cuda.reset_peak_memory_stats()
        n_lotes=len(loader_train)
        for paso,lote in enumerate(tqdm(loader_train,desc=f'{tag} | época {epoca}/{epocas}',leave=False),start=1):
            lote={k:v.to(DEVICE) for k,v in lote.items()}
            y=lote.pop('labels')
            # Mantener escala correcta en un último grupo de acumulación incompleto.
            grupo_actual=min(GRAD_ACCUM,n_lotes-((paso-1)//GRAD_ACCUM)*GRAD_ACCUM)
            with contexto_amp():
                logits=modelo(**lote).logits
                loss_original=F.cross_entropy(logits.float(),y)
                loss=loss_original/grupo_actual
            scaler.scale(loss).backward()
            if paso%GRAD_ACCUM==0 or paso==n_lotes:
                scaler.unscale_(optimizador)
                torch.nn.utils.clip_grad_norm_(modelo.parameters(),1.0)
                scaler.step(optimizador)
                scaler.update()
                scheduler.step()
                optimizador.zero_grad(set_to_none=True)
            suma_loss+=float(loss_original.detach().item())*len(y)
            total+=len(y)
        registro={'epoca':epoca,'loss_entrenamiento':suma_loss/total}
        if loader_val is not None:
            met=evaluar_epoca(modelo,loader_val)
            registro.update({'accuracy_validacion':met['accuracy'],'loss_validacion':met['loss']})
            mejora=(met['accuracy']>mejor+1e-7 or (abs(met['accuracy']-mejor)<1e-7 and met['loss']<mejor_loss-1e-5))
            if mejora:
                mejor,mejor_loss,mejor_epoca=met['accuracy'],met['loss'],epoca
                guardar_pesos_cpu(modelo,tokenizer,checkpoint)
                sin_mejora=0
            else:
                sin_mejora+=1
        else:
            mejor_epoca=epoca
        registro['memoria_pico_gpu_gb']=round(torch.cuda.max_memory_allocated()/1024**3,2)
        historial.append(registro)
        print(tag,registro,flush=True)
        if loader_val is not None and parada_temprana and sin_mejora>=PATIENCE:
            print('Parada temprana: no hubo mejora reciente.')
            break
    if loader_val is None:
        guardar_pesos_cpu(modelo,tokenizer,checkpoint)
    resumen={
        'tag':tag,'base':base,'aumentacion':bool(aumentar),'epocas_configuradas':epocas,
        'mejor_epoca':mejor_epoca,'accuracy_validacion':None if loader_val is None else float(mejor),
        'checkpoint':str(checkpoint),'historial':historial,'parametros':clave_parametros,
    }
    resumen_path.write_text(json.dumps(resumen,ensure_ascii=False,indent=2),encoding='utf-8')
    pd.DataFrame(historial).to_csv(carpeta/'historial.csv',index=False)
    del modelo,tokenizer,optimizador,scheduler,scaler,loader_train,loader_val,ds_train
    liberar_gpu()
    return resumen


## 11. Experimento A: mDeBERTa multilingüe, sin aumentación

Se ajusta `microsoft/mdeberta-v3-base`, un transformer multilingüe preentrenado (licencia MIT), sobre la partición de entrenamiento original. **No se generan ejemplos sintéticos**. Su accuracy y su pérdida validación constituyen el grupo de control para la prueba de aumentación. Se conservan los mejores pesos, no solo el resultado de la última época.

In [ ]:
experimentos = {}
experimentos['mdeberta_sin_aug'] = ajustar_transformer(
    BASE_MDEBERTA, 'mdeberta_sin_aug', sub_train, sub_val,
    aumentar=False, epocas=MAX_EPOCHS,
)

## 12. Experimento B: el mismo mDeBERTa, con aumentación

Se repite el experimento A con **el mismo modelo base, la misma semilla, los mismos datos reales, la misma partición y el mismo número máximo de épocas**, pero habilitando `aug_superficial` en el 40 % de los ejemplos de cada época. De esta comparación se calcula de forma directa la ganancia o pérdida de accuracy debida a la aumentación, sin confundirla con el cambio de arquitectura.

In [ ]:
experimentos['mdeberta_con_aug'] = ajustar_transformer(
    BASE_MDEBERTA, 'mdeberta_con_aug', sub_train, sub_val,
    aumentar=True, epocas=MAX_EPOCHS,
)

## 13. Experimento C: XLM-RoBERTa especializado en sentimientos, con aumentación

Se transfiere `clapAI/roberta-base-multilingual-sentiment`, basado en XLM-RoBERTa y previamente entrenado para clasificación positiva, neutral y negativa en varios idiomas (licencia Apache 2.0). Su cabeza original declara el orden `negative, neutral, positive`, equivalente al utilizado aquí; por ello sus pesos se pueden reutilizar. Se realiza *fine-tuning* con aumentación superficial y se compara contra los dos experimentos anteriores.

Este candidato incorpora conocimiento de sentimientos previo al ajuste de las reseñas del curso, lo que puede resultar útil con sarcasmo, inferencias implícitas y muestras multilingües. **La utilidad real se establece con validación, no con esta hipótesis.**

In [ ]:
experimentos['xlmr_sentimiento_aug'] = ajustar_transformer(
    BASE_SENTIMIENTO, 'xlmr_sentimiento_aug', sub_train, sub_val,
    aumentar=True, epocas=MAX_EPOCHS,
)

## 14. Cuantificación de las iteraciones y del efecto de aumentación

Se reconstruyen los mejores checkpoints de los tres experimentos, se obtienen las probabilidades sobre la **misma validación** y se presenta la tabla de accuracies. La diferencia entre A y B es el efecto experimental de la aumentación. También se dibujan curvas de aprendizaje para distinguir mejora genuina de sobreajuste. Estos valores **solo aparecen tras ejecutar el entrenamiento**; no se consignan resultados hipotéticos.

In [ ]:
prob_val = {}
filas = []
for tag, exp in experimentos.items():
    prob_val[tag] = predecir_probabilidades(exp['checkpoint'], sub_val)
    accuracy = accuracy_score(y_val, prob_val[tag].argmax(axis=1))
    filas.append({'modelo':tag, 'base':exp['base'], 'aumentacion':exp['aumentacion'],
                  'mejor_epoca':exp['mejor_epoca'], 'accuracy_validacion':accuracy})
resumen_experimentos = pd.DataFrame(filas).sort_values('accuracy_validacion', ascending=False)
display(resumen_experimentos)
ganancia_aumentacion = (
    resumen_experimentos.set_index('modelo').loc['mdeberta_con_aug', 'accuracy_validacion']
    - resumen_experimentos.set_index('modelo').loc['mdeberta_sin_aug', 'accuracy_validacion']
)
print(f'Diferencia CON - SIN aumentación (mDeBERTa): {ganancia_aumentacion:+.4f} puntos absolutos')

fig, ax = plt.subplots(figsize=(9, 4))
for tag, exp in experimentos.items():
    hist = pd.DataFrame(exp['historial'])
    ax.plot(hist['epoca'], hist['accuracy_validacion'], marker='o', label=tag)
ax.set(title='Accuracy de validación por época', xlabel='Época', ylabel='Accuracy', ylim=(0, 1))
ax.legend(loc='lower right')
ax.grid(alpha=.25)
plt.show()
resumen_experimentos.to_csv(OUTPUT_DIR/'tabla_experimentos.csv', index=False)

## 15. Evaluación por idioma, matrices de confusión y errores concretos

Se evalúa cada modelo por subgrupo (`es`, `en`, `fr`, `pt`, `es_en_mixto` cuando exista) y por clase. La etiqueta de idioma se calcula con una heurística y, por tanto, el análisis describe **grupos estimados**, no una verdad lingüística absoluta. Las matrices de confusión y las reseñas peor clasificadas permiten examinar si los errores se concentran en negaciones, concesiones o mezclas de idiomas. Se muestran textos completos para que las decisiones sean defendibles en la sustentación.

In [ ]:
registros_idioma = []
for tag, probs in prob_val.items():
    pred = probs.argmax(axis=1)
    for idioma, idx in sub_val.groupby('idioma').groups.items():
        idx = np.asarray(list(idx), dtype=int)
        registros_idioma.append({'modelo':tag, 'idioma':idioma,
                                 'n':len(idx), 'accuracy':accuracy_score(y_val[idx], pred[idx])})
resultados_idioma = pd.DataFrame(registros_idioma)
display(resultados_idioma.pivot(index='idioma', columns='modelo', values='accuracy').round(4))
display(resultados_idioma.pivot(index='idioma', columns='modelo', values='n').iloc[:, :1])
resultados_idioma.to_csv(OUTPUT_DIR/'errores_por_idioma.csv', index=False)

mejor_individual = resumen_experimentos.iloc[0].modelo
probs_individual = prob_val[mejor_individual]
pred_individual = probs_individual.argmax(axis=1)
matriz = confusion_matrix(y_val, pred_individual, labels=[0,1,2])
fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(matriz, cmap='Blues')
for i in range(3):
    for j in range(3):
        ax.text(j, i, matriz[i,j], ha='center', va='center')
ax.set(xticks=range(3), yticks=range(3), xticklabels=LABELS, yticklabels=LABELS,
       title=f'Confusión: {mejor_individual}', xlabel='Predicho', ylabel='Real')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()
print(classification_report(y_val, pred_individual, target_names=LABELS))

errores = sub_val.copy()
errores['real'] = errores['label']
errores['predicho'] = [ID2LABEL[i] for i in pred_individual]
errores['confianza'] = probs_individual.max(axis=1)
errores = errores[errores.real != errores.predicho].sort_values('confianza', ascending=False)
display(errores[['idioma', 'real', 'predicho', 'confianza', 'text']].head(15))

## 16. Selección y prueba de ensambles sobre validación

La predicción de una fusión se define como promedio ponderado de **probabilidades** de transformers ya entrenados. Se consideran únicamente combinaciones **preespecificadas** (promedios entre dos modelos, los tres modelos y una ponderación hacia el candidato con sentimientos previos), para reducir el riesgo de ajustar arbitrariamente muchos pesos al pequeño conjunto de validación. La selección es por accuracy validada; los empates favorecen menor número de modelos, más eficiente de entregar y sustentar.

Los ensambles no introducen datos externos, etiquetas nuevas ni modelos generativos. Si un ensamble es el ganador, el paquete final conserva todos sus componentes y los coeficientes necesarios para reproducirlo.

In [ ]:
W = {
    'mdeberta_sin_aug':  {'mdeberta_sin_aug':1.0},
    'mdeberta_con_aug':  {'mdeberta_con_aug':1.0},
    'xlmr_sentimiento_aug':{'xlmr_sentimiento_aug':1.0},
    'ensamble_mdeberta': {'mdeberta_sin_aug':0.50, 'mdeberta_con_aug':0.50},
    'ensamble_2': {'mdeberta_con_aug':0.50, 'xlmr_sentimiento_aug':0.50},
    'ensamble_3': {'mdeberta_sin_aug':1/3, 'mdeberta_con_aug':1/3, 'xlmr_sentimiento_aug':1/3},
    'ensamble_especializado':{'mdeberta_con_aug':0.35, 'xlmr_sentimiento_aug':0.65},
}

def fusionar(probs_por_modelo, pesos):
    assert abs(sum(pesos.values()) - 1.0) < 1e-8
    resultado = sum(float(w) * probs_por_modelo[k] for k,w in pesos.items())
    return resultado / resultado.sum(axis=1, keepdims=True)

candidatos = []
for nombre, pesos in W.items():
    scores = fusionar(prob_val, pesos)
    candidatos.append({'candidato':nombre, 'componentes':len(pesos),
                       'accuracy_validacion':accuracy_score(y_val, scores.argmax(axis=1)),
                       'logloss_validacion':log_loss(y_val, scores, labels=list(range(3)))})
tabla_candidatos = pd.DataFrame(candidatos).sort_values(
    ['accuracy_validacion', 'componentes', 'logloss_validacion'],
    ascending=[False, True, True], kind='stable',
).reset_index(drop=True)
display(tabla_candidatos)
tabla_candidatos.to_csv(OUTPUT_DIR/'tabla_candidatos.csv', index=False)
GANADOR = tabla_candidatos.iloc[0]['candidato']
PESOS_FINALES = W[GANADOR]
print('GANADOR VALIDACIÓN:', GANADOR)
print('Pesos:', PESOS_FINALES)
print('Accuracy estimado validación:', float(tabla_candidatos.iloc[0]['accuracy_validacion']))
print('Meta de 0,91 alcanzada en validación:', bool(tabla_candidatos.iloc[0]['accuracy_validacion'] >= 0.91))

## 17. Diagnóstico del modelo elegido

Tras seleccionar la combinación, se vuelve a inspeccionar su error por idioma, clasificación y confianza. Esto responde a la rúbrica sobre análisis de fallas de los modelos multilingües. Los casos concretos mostrados aquí siguen perteneciendo exclusivamente a validación y, por tanto, **sí tienen** etiquetas reales conocidas. No se presenta una supuesta medición del accuracy de `eval.csv`, pues esas etiquetas son ocultas.

In [ ]:
prob_ganador = fusionar(prob_val, PESOS_FINALES)
pred_ganador = prob_ganador.argmax(axis=1)
print('Accuracy validación final:', accuracy_score(y_val, pred_ganador))
print(classification_report(y_val, pred_ganador, target_names=LABELS, digits=4))
por_idioma_ganador = []
for idioma, ids in sub_val.groupby('idioma').groups.items():
    ids = np.asarray(list(ids), dtype=int)
    por_idioma_ganador.append({'idioma':idioma, 'n':len(ids),
                               'accuracy':accuracy_score(y_val[ids], pred_ganador[ids])})
display(pd.DataFrame(por_idioma_ganador).sort_values('accuracy'))
fallos_finales = sub_val.copy()
fallos_finales['prediccion'] = [ID2LABEL[i] for i in pred_ganador]
fallos_finales['confianza'] = prob_ganador.max(axis=1)
fallos_finales = fallos_finales[fallos_finales['label'] != fallos_finales['prediccion']]
display(fallos_finales.sort_values('confianza', ascending=False)[
    ['idioma','label','prediccion','confianza','text']].head(15))

## 18. Reentrenamiento final con `train.csv` completo

El modelo ganador ya permite inferir con pesos optimizados en `sub_train` y seleccionados mediante `sub_val`. El parámetro REENTRENAR_FINAL se puede activar `True` y ejecutar desde esta celda después de haber seleccionado el mejor ensamble; se entrenan sus componentes desde sus bases originales durante el número de épocas elegido en validación. En ningún caso se entrena con `eval.csv`.



In [ ]:
CHECKPOINTS_FINALES = {}
for tag in PESOS_FINALES:
    exp = experimentos[tag]
    if REENTRENAR_FINAL:
        nuevo = ajustar_transformer(
            base=exp['base'], tag=tag, df_ajuste=train_df,
            df_validacion=None, aumentar=exp['aumentacion'],
            epocas=int(exp['mejor_epoca']), parada_temprana=False,
        )
        CHECKPOINTS_FINALES[tag] = nuevo['checkpoint']
    else:
        CHECKPOINTS_FINALES[tag] = exp['checkpoint']
print('Modelos que componen la entrega final:')
for k,v in CHECKPOINTS_FINALES.items():
    print(' ', k, '->', v)

## 19. Predicción de `eval.csv` y archivo `submission.csv`

Se aplica inferencia sobre cada reseña de evaluación, obteniendo probabilidades del mejor modelo o ensamble. Se convierten las clases numéricas al español, conservando el orden de `eval.csv`. Se guarda submission.csv en /content/salidas/ y /content/.


In [ ]:
prob_eval_finales = {}
for tag, checkpoint in CHECKPOINTS_FINALES.items():
    prob_eval_finales[tag] = predecir_probabilidades(checkpoint, eval_df)
probs_eval = fusionar(prob_eval_finales, PESOS_FINALES)
labels_eval = [ID2LABEL[int(i)] for i in probs_eval.argmax(axis=1)]
submission = pd.DataFrame({'id':eval_df['id'].to_numpy(), 'answer':labels_eval})
assert submission.shape == (len(eval_df), 2)
assert submission.id.is_unique and submission.answer.isin(LABELS).all()
assert submission['id'].equals(plantilla['id'])
assert not submission.isna().any().any()
archivo_submission = OUTPUT_DIR/'submission.csv'
submission.to_csv(archivo_submission, index=False, encoding='utf-8')
submission.to_csv(Path.cwd()/'submission.csv', index=False, encoding='utf-8')
print('Archivo Kaggle:', archivo_submission)
display(submission.head(10))
print('Distribución de predicciones:', submission.answer.value_counts().to_dict())

## 20. Archivos alternativos para documentar la iteración en Kaggle

La rúbrica requiere al menos **cinco envíos diferentes** y superar la línea base del leaderboard privado. Se crean variantes reproducibles de los modelos y ensambles probados sobre la misma validación. Algunas variantes pueden predecir exactamente lo mismo para `eval`; por eso se eliminan los archivos duplicados y **no se afirma** haber cumplido cinco entregas si no se generaron cinco predicciones distintas. Solo la plataforma Kaggle confirma que fueron enviados.


In [ ]:
prob_eval_experimentos = dict(prob_eval_finales)
for tag, exp in experimentos.items():
    if tag not in prob_eval_experimentos:
        prob_eval_experimentos[tag] = predecir_probabilidades(exp['checkpoint'], eval_df)

carpeta_variantes = OUTPUT_DIR/'submissions_iteraciones'
carpeta_variantes.mkdir(parents=True, exist_ok=True)
propuestas = []
huellas = set()
for nombre, pesos in W.items():
    pred = fusionar(prob_eval_experimentos, pesos).argmax(axis=1)
    huella = pred.tobytes()
    if huella in huellas:
        continue
    huellas.add(huella)
    ruta = carpeta_variantes/f'{nombre}.csv'
    pd.DataFrame({'id':eval_df.id, 'answer':[ID2LABEL[int(i)] for i in pred]}).to_csv(ruta, index=False)
    propuestas.append({'variante':nombre, 'archivo':str(ruta),
                       'accuracy_validacion':float(tabla_candidatos.set_index('candidato').loc[nombre, 'accuracy_validacion'])})
display(pd.DataFrame(propuestas).sort_values('accuracy_validacion', ascending=False))
print('Número de propuestas distintas:', len(propuestas))
if len(propuestas) < 5:
    print('No se generaron cinco predicciones realmente distintas; se necesitan otras iteraciones legítimas.')

## 21. Artefactos, referencias y reproducibilidad

Los checkpoints completos (pesos, configuración y tokenizador) quedan en `salidas/experimentos` o `salidas/modelo_final`. Para simplificar la primera ejecución, se crea por defecto un archivo ZIP con los pesos (`EMPAQUETAR_MODELO=True` en la configuración); los checkpoints originales también quedan en `/content/salidas`, pero **son temporales**. Se genera un manifiesto con parámetros, modelos preentrenados, licencias y score de validación.

Esto satisface el requisito de entregar el modelo entrenado con bibliotecas vistas en clase, siempre que el equipo descargue y conserve el ZIP o los checkpoints correspondientes antes de cerrar Colab. El archivo `submission.csv` es la entrega independiente para la competencia.


In [ ]:
DIRECTORIO_ENTREGA = OUTPUT_DIR/'entrega_modelo_final'
DIRECTORIO_ENTREGA.mkdir(parents=True,exist_ok=True)

manifiesto={
    'modelo_ganador':GANADOR,
    'pesos':PESOS_FINALES,
    'componentes':{
        tag:{'directorio_checkpoint':str(CHECKPOINTS_FINALES[tag]),'base':experimentos[tag]['base'],
             'con_aumentacion':experimentos[tag]['aumentacion'],
             'epocas_finales':experimentos[tag]['mejor_epoca']}
        for tag in PESOS_FINALES
    },
    'orden_labels':LABELS,
    'accuracy_validacion_antes_refit':float(tabla_candidatos.iloc[0]['accuracy_validacion']),
    'reentrenado_con_train_completo':REENTRENAR_FINAL,
    'semilla':SEED,'max_tokens':MAX_LENGTH,'augmentation_probability':AUG_PROB,
    'train_records':len(train_df),'eval_records':len(eval_df),
    'versiones':{'torch':torch.__version__,'transformers':importlib.metadata.version('transformers'),
                'scikit-learn':sklearn.__version__},
    'origen_y_licencia':{
        BASE_MDEBERTA:{'url':'https://huggingface.co/microsoft/mdeberta-v3-base','licencia':'MIT'},
        BASE_SENTIMIENTO:{'url':'https://huggingface.co/clapAI/roberta-base-multilingual-sentiment','licencia':'Apache-2.0'},
    },
}
(DIRECTORIO_ENTREGA/'manifiesto.json').write_text(json.dumps(manifiesto,ensure_ascii=False,indent=2),encoding='utf-8')
zip_path=None
if EMPAQUETAR_MODELO:
    temporal = Path('/content/modelo_para_entregar')
    if temporal.exists():
        shutil.rmtree(temporal)
    temporal.mkdir(parents=True)
    shutil.copy2(DIRECTORIO_ENTREGA/'manifiesto.json',temporal/'manifiesto.json')
    for tag,checkpoint in CHECKPOINTS_FINALES.items():
        shutil.copytree(checkpoint,temporal/tag)
    zip_path=Path(shutil.make_archive(str(OUTPUT_DIR/'modelo_final'),'zip',root_dir=temporal))
    print('Pesos empaquetados:',zip_path)
    shutil.rmtree(temporal)
else:
    print('Checkpoint(s) conservados temporalmente en /content:')
    for k,v in CHECKPOINTS_FINALES.items():
        print('  ',k,'→',v)
    print('AVISO: descarga estos archivos antes de desconectar Colab. Si necesitas un ZIP, activa EMPAQUETAR_MODELO=True y ejecuta esta celda.')
print('Manifiesto:',DIRECTORIO_ENTREGA/'manifiesto.json')
print('Archivo Kaggle:',archivo_submission)


## 22. Comprobaciones antes de Kaggle

Se verifica que las 6.781 predicciones estén presentes, que sus identificadores estén en el orden de la plantilla y que sus clases pertenezcan a `negativo`, `neutral`, `positivo`. También se comprueba la existencia de los archivos de pesos y tokenización de todos los componentes del modelo final. Después de la comprobación se puede descargar el CSV directamente mediante la última celda.


In [ ]:
chequeo=pd.read_csv(archivo_submission)
assert list(chequeo.columns)==['id','answer']
assert len(chequeo)==len(eval_df)==len(plantilla)==6781
assert chequeo.id.equals(eval_df.id)
assert chequeo.answer.isin(LABELS).all()
assert chequeo.id.nunique()==len(chequeo)
for nombre,checkpoint in CHECKPOINTS_FINALES.items():
    carpeta=Path(checkpoint)
    assert (carpeta/'config.json').is_file(), nombre
    assert (carpeta/'tokenizer_config.json').is_file(), nombre
    assert any((carpeta/f).is_file() for f in ('model.safetensors','pytorch_model.bin')), nombre
if EMPAQUETAR_MODELO:
    assert zip_path and zip_path.exists()
print('VALIDACIÓN DEL FORMATO Y LOS CHECKPOINTS: CORRECTA')
print('Accuracy de validación antes del refit (NO es score de Kaggle):',
      f"{manifiesto['accuracy_validacion_antes_refit']:.4f}")
print('Archivo listo para subir:',archivo_submission)


## 23. Fuentes, licencias y declaración del uso de IA generativa

**Datos del curso:** `train.csv`, `eval.csv` y `sample_submission.csv` proporcionados por el equipo docente. No se añaden datos etiquetados de terceros. El conjunto de prueba `eval.csv` se mantiene aislado de entrenamiento y ajuste.

**Modelos preentrenados:**
- Microsoft, [mDeBERTa-v3-base](https://huggingface.co/microsoft/mdeberta-v3-base), licencia **MIT**.
- clapAI, [roberta-base-multilingual-sentiment](https://huggingface.co/clapAI/roberta-base-multilingual-sentiment), licencia **Apache-2.0**; basado en [FacebookAI/xlm-roberta-base](https://huggingface.co/FacebookAI/xlm-roberta-base) (MIT).
- Hugging Face [Transformers](https://github.com/huggingface/transformers), Apache-2.0, y [PyTorch](https://pytorch.org/), BSD-3-Clause.
- Shuyo, [langdetect](https://pypi.org/project/langdetect/), Apache-2.0. La etiqueta de idioma es orientativa y no es un objetivo de entrenamiento.

**Aumentación de datos:** generación en memoria de variaciones ortográficas de reseñas del entrenamiento mediante `aug_superficial`; no se generan etiquetas externas ni reseñas de terceros. Semilla y probabilidad de alteración aparecen en el código.

**Uso de IA generativa:** ChatGPT se utilizó como apoyo de programación, documentación y revisión del flujo experimental. No se utilizó para asignar etiquetas directamente a los textos de `eval.csv`. El equipo debe revisar, completar esta declaración y sustentar las decisiones efectivamente ejecutadas.

**Colab:** [notebooks de Google Colab](https://colab.research.google.com/), [limitaciones y comportamiento del entorno](https://research.google.com/colaboratory/faq.html). Los recursos gratuitos pueden agotarse y la disponibilidad de GPU no está garantizada.



## 24. Guardado y descarga del modelo entrenado

En esta sección se exporta el modelo final obtenido mediante transferencia de aprendizaje y entrenamiento con PyTorch. Se guardan los pesos de los transformers, sus configuraciones, tokenizadores y los pesos de combinación del ensamble, cuando corresponda.

El modelo se empaqueta en un archivo ZIP para permitir su descarga y posterior carga sin necesidad de repetir el entrenamiento. Este archivo constituye el entregable del modelo entrenado solicitado en el proyecto.

In [ ]:

import json
import zipfile
from pathlib import Path
from google.colab import files

carpeta_salida = Path("/content/salidas")
archivo_modelo = carpeta_salida / "modelo_entrenado.zip"

if not CHECKPOINTS_FINALES:
    raise RuntimeError("Primero debes ejecutar el entrenamiento y la selección del modelo final.")

manifiesto = {
    "modelo_ganador": GANADOR,
    "pesos_ensamble": PESOS_FINALES,
    "etiquetas": LABELS,
    "max_length": MAX_LENGTH,
    "framework": "PyTorch",
    "modelos": list(CHECKPOINTS_FINALES.keys())
}

with zipfile.ZipFile(
    archivo_modelo, "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=1
) as zipf:

    zipf.writestr(
        "manifiesto.json",
        json.dumps(manifiesto, indent=4, ensure_ascii=False)
    )

    for nombre, ruta in CHECKPOINTS_FINALES.items():
        carpeta_modelo = Path(ruta)

        if not carpeta_modelo.exists():
            raise FileNotFoundError(
                f"No se encontró el modelo: {carpeta_modelo}"
            )

        for archivo in carpeta_modelo.rglob("*"):
            if archivo.is_file():
                ruta_interna = (
                    Path(nombre) /
                    archivo.relative_to(carpeta_modelo)
                )
                zipf.write(archivo, ruta_interna)

print("Modelo guardado correctamente.")
print("Archivo:", archivo_modelo)
print("Tamaño:", round(archivo_modelo.stat().st_size / 1024**2, 2), "MB")
print("Modelos incluidos:", list(CHECKPOINTS_FINALES.keys()))

files.download(str(archivo_modelo))
